In [1]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam


In [2]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    preprocessing_function=None,   # EfficientNet uses rescale
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    "/Users/kpvarma/PycharmProjects/Chest_X_Rays_Pnemonia_3/Dataset/train",
    target_size=(224,224),
    batch_size=16,
    class_mode="categorical"
)

val_gen = val_datagen.flow_from_directory(
    "/Users/kpvarma/PycharmProjects/Chest_X_Rays_Pnemonia_3/Dataset/val",
    target_size=(224,224),
    batch_size=16,
    class_mode="categorical"
)


Found 4835 images belonging to 3 classes.
Found 1031 images belonging to 3 classes.


In [3]:
base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

In [4]:
for layer in base_model.layers:
    layer.trainable = False


In [5]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)
output = Dense(3, activation="softmax")(x)   # covid, normal, pneumonia

model = Model(inputs=base_model.input, outputs=output)

In [6]:
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history = model.fit(
    train_gen,
    epochs=15,
    validation_data=val_gen
)


Epoch 1/15
113/303 ━━━━━━━━━━━━━━━━━━━━ 35s 186ms/step - accuracy: 0.3299 - loss: 1.1169

In [ ]:
test_gen = val_datagen.flow_from_directory(
    "/Users/kpvarma/PycharmProjects/Chest_X_Rays_Pnemonia_3/Dataset/test",
    target_size=(224,224),
    batch_size=16,
    class_mode="categorical",
    shuffle=False
)

model.evaluate(test_gen)


In [ ]:
for layer in base_model.layers[-30:]:
    layer.trainable = True

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.fit(
    train_gen,
    epochs=10,
    validation_data=val_gen
)
